# Raw Reaction Model Bakeoff Control Panel

Use this notebook to run the full experiment from one place: freeze tasks, run candidate generation, run the strong-model gold audit, score results, and inspect visuals.

Live model calls are behind explicit switches in the setup cell. Start with a smoke run before spending on the full configuration.

## 1. Setup

In [1]:
from pathlib import Path
from pprint import pprint
import json
import os
import sys

cwd = Path.cwd().resolve()
REPO_ROOT = cwd
while REPO_ROOT.name != "chemy" and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from experiments.model_quality.helpers import load_config, read_jsonl
from experiments.model_quality.sample import build_tasks
from experiments.model_quality.run_generation import run_generation
from experiments.model_quality.run_gold_audit import run_gold_audit
from experiments.model_quality.score import build_summary

CONFIG_PATH = REPO_ROOT / "experiments" / "model_quality" / "config.yaml"

# Safety switches. Keep live steps False until you intentionally want to call models.
RUN_FREEZE_TASKS = False
RUN_GENERATION = False
RUN_GOLD_AUDIT = False
RUN_SCORE = True

# Smoke mode overrides the loaded config in-memory only. It does not edit config.yaml.
SMOKE_MODE = True
SMOKE_MODELS = ["openai/gpt-oss-120b"]
SMOKE_PRESETS = ["wiki_crc_rp"]
SMOKE_COMPLEXITY_BANDS = ["simple"]
SMOKE_GENERATION_RUN_LIMIT = 2
SMOKE_MAX_WORKERS = 2

DATA_DIR = REPO_ROOT / "experiments" / "model_quality" / ("data_smoke" if SMOKE_MODE else "data")
SUMMARY_PATH = DATA_DIR / "summary.json"

config = load_config(CONFIG_PATH)
if SMOKE_MODE:
    config = {
        **config,
        "tasks_per_band": 1,
        "gold_rounds": 1,
        "candidate_models": SMOKE_MODELS,
        "bakeoff_presets": SMOKE_PRESETS,
        "complexity_bands": SMOKE_COMPLEXITY_BANDS,
        "generation_self_review": False,
        "max_workers": SMOKE_MAX_WORKERS,
    }
GENERATION_RUN_LIMIT = SMOKE_GENERATION_RUN_LIMIT if SMOKE_MODE else None

print(f"Repo root: {REPO_ROOT}")
print(f"Config:    {CONFIG_PATH}")
print(f"Data dir:  {DATA_DIR}")
print(f"API key present: {bool(os.getenv('OPENROUTER_API_KEY'))}")
print(f"Live switches: freeze={RUN_FREEZE_TASKS}, generation={RUN_GENERATION}, gold={RUN_GOLD_AUDIT}, score={RUN_SCORE}")
print(f"Generation run limit: {GENERATION_RUN_LIMIT}")
print(f"Max workers: {config['max_workers']}")
pprint(config)

Repo root: /home/canary/Documents/Code/chemy
Config:    /home/canary/Documents/Code/chemy/experiments/model_quality/config.yaml
Data dir:  /home/canary/Documents/Code/chemy/experiments/model_quality/data
API key present: True
Live switches: freeze=True, generation=True, gold=True, score=True
{'acceptance_threshold': 0.4,
 'bakeoff_presets': ['wiki_crc_rp',
                     'simple_inorganic_wiki_rp',
                     'simple_organic_wiki_rp',
                     'oxidizers_wiki_rp',
                     'corrosive_wiki_rp',
                     'top_rare_rp',
                     'wiki_uncommon_p'],
 'batch_size': 5,
 'candidate_models': ['openai/gpt-oss-120b'],
 'chemy_data_dir': 'data',
 'complexity_bands': ['simple', 'medium', 'hard'],
 'generation_self_review': True,
 'gold_rounds': 1,
 'seed': 1729,
 'strong_model': 'google/gemini-2.5-pro',
 'tasks_per_band': 1}


## 2. Freeze Tasks

In [ ]:
if RUN_FREEZE_TASKS:
    tasks_path, task_manifest = build_tasks(config, data_dir=DATA_DIR, root=REPO_ROOT)
    print(f"Wrote {task_manifest['task_n']} tasks to {tasks_path}")
else:
    task_manifest_path = DATA_DIR / "manifests" / "task_manifest.json"
    task_manifest = json.loads(task_manifest_path.read_text()) if task_manifest_path.exists() else None
    print("Task freezing skipped.")

tasks = read_jsonl(DATA_DIR / "tasks.jsonl")
print(f"Frozen task rows: {len(tasks)}")
tasks[:3]

Wrote 21 tasks to /home/canary/Documents/Code/chemy/experiments/model_quality/data/tasks.jsonl
Frozen task rows: 21


[{'task_id': 'task:9e2807a7382066c261b1f7f55ddef22ddd5a09a4',
  'cid': 14827,
  'compound_name': 'Lead monoxide',
  'preset': 'wiki_crc_rp',
  'position': 'any',
  'scope': 'documented_less_common',
  'prompt': 'Provide a list of documented chemical reactions involving Lead monoxide, where it appears as a reagent or product. Include not only the most common reactions, but also less common or unusual ones, as long as you are absolutely sure they are real and correct. Return one JSON object per line (JSONL). Each JSON object has these fields:\n- "reagents": list of {"name": "<clean chemical name>", "phase": "<g|l|s|aq>"}\n- "products": list of {"name": "<clean chemical name>", "phase": "<g|l|s|aq>"}\n- Optional "solvent": primary solvent name or null\n- Optional "catalyst": catalyst name or null\n- Optional per-compound "note": short qualifier; omit it when not needed\n\nRules:\n- Use full chemical names, not formulas.\n- The name field must contain ONLY the chemical name.\n- Put qualifi

## 3. Run Candidate Generation

In [ ]:
if RUN_GENERATION:
    generation_manifest = run_generation(
        config,
        data_dir=DATA_DIR,
        models=config["candidate_models"],
        run_limit=GENERATION_RUN_LIMIT,
        max_workers=config["max_workers"],
    )
    pprint(generation_manifest)
else:
    print("Generation skipped.")

generations = read_jsonl(DATA_DIR / "generations.jsonl")
candidates = read_jsonl(DATA_DIR / "candidates.jsonl")
print(f"Generation rows: {len(generations)}")
print(f"Candidate rows:  {len(candidates)}")
candidates[:3]

## 4. Run Gold Audit

In [ ]:
if RUN_GOLD_AUDIT:
    gold_manifest = run_gold_audit(
        config,
        data_dir=DATA_DIR,
        model=config["strong_model"],
        max_workers=config["max_workers"],
    )
    pprint(gold_manifest)
else:
    print("Gold audit skipped.")

gold_audits = read_jsonl(DATA_DIR / "gold_audits.jsonl")
print(f"Gold audit rows: {len(gold_audits)}")
gold_audits[:3]

## 5. Score

In [ ]:
summary = build_summary(config, data_dir=DATA_DIR)
if RUN_SCORE:
    SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)
    SUMMARY_PATH.write_text(json.dumps(summary, indent=2) + "\n")
    print(f"Wrote {SUMMARY_PATH}")

summary["inputs"], summary["models"]

## 6. Visuals

In [ ]:
import matplotlib.pyplot as plt

models = list(summary["models"])
if not models:
    print("No scored models yet. Run generation and gold audit first.")
else:
    precision = [summary["models"][m]["gold_precision"] for m in models]
    yield_ = [summary["models"][m]["gold_accepted_unique_rid_n"] for m in models]

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(yield_, precision)
    for m, x, y in zip(models, yield_, precision):
        ax.annotate(m, (x, y), xytext=(4, 4), textcoords="offset points")
    ax.set_xlabel("Gold-accepted unique rid count")
    ax.set_ylabel("Gold precision")
    ax.set_title("Precision vs accepted unique yield")
    ax.grid(True, alpha=0.25)

In [ ]:
metrics = ["schema_success_rate", "parsed_rate", "target_position_rate", "balance_success_rate"]
if models:
    fig, axes = plt.subplots(1, len(metrics), figsize=(4 * len(metrics), 4), sharey=True)
    for ax, metric in zip(axes, metrics):
        ax.bar(models, [summary["models"][m][metric] for m in models])
        ax.set_title(metric)
        ax.tick_params(axis="x", rotation=45)
        ax.set_ylim(0, 1)
    fig.tight_layout()

In [ ]:
summary["bands"]